# Run MPI Training with Kubeflow Trainer v2

Run an OpenMPI workload with Kubeflow Trainer v2 using a `TrainingRuntime` and a `TrainJob`. Trainer v2 creates a JobSet to orchestrate the launcher and worker pods. You submit a `TrainJob`; the older `MPIJob` custom resource is not the API used by Trainer v2.

This notebook starts a two-node MPI job that calculates an approximation of pi. It uses the MPI Operator sample image, which includes OpenMPI, the SSH setup required by the Trainer MPI runtime, and the MPI program. Replace the sample program with your own MPI application after the sample job succeeds.

## Prerequisites

- Kubeflow Trainer v2 installed with MPI policy support. Confirm the cluster serves `TrainingRuntime`, `TrainJob`, and `JobSet` resources.
- Permission to create `TrainingRuntime` and `TrainJob` resources in your namespace.
- Two schedulable CPU nodes and permission for the runtime image to be pulled from Docker Hub.
- A namespace that allows the runtime's non-root MPI user to run. Adjust the pod security settings to match your cluster policy.

Trainer's MPI policy currently uses OpenMPI. The example pins the multi-architecture sample image to `v0.6.0-openmpi`; use a compatible, pinned image when building your own MPI environment.

## Set the namespace

Run this notebook from `docs/en/train/guides` in an `aml-docs` checkout. The runtime and job manifests are in the adjacent `assets/` directory.

In [ ]:
import os

NAMESPACE = os.environ.get("NAMESPACE", "finetune")
print(f"Using namespace: {NAMESPACE}")

## Create an MPI training runtime

The runtime is a reusable template. Its `node` job starts the SSH daemon used by OpenMPI. Trainer adds the launcher job and generates the hostfile from `numNodes` and `numProcPerNode`. `runLauncherAsNode: true` counts the launcher as one of the training nodes.

The runtime needs a headless network with not-ready addresses published so the launcher can resolve worker pods before they become ready. The node image must include the matching OpenMPI and SSH setup, and use the SSH key mount path configured in `mlPolicy.mpi.sshAuthMountPath`.

In [ ]:
!kubectl apply -n {NAMESPACE} -f assets/mpi-trainer-v2/openmpi-trainingruntime.yaml
!kubectl get trainingruntime openmpi-trainingruntime -n {NAMESPACE}

## Submit a two-node MPI job

This job starts two MPI ranks and prints each rank and hostname while calculating pi. The MPI plugin supplies the hostfile, so the command does not need an explicit `-np` argument.

In [ ]:
!kubectl apply -n {NAMESPACE} -f assets/mpi-trainer-v2/trainjob-hello-mpi.yaml
!kubectl wait --for=condition=Complete trainjob/hello-mpi -n {NAMESPACE} --timeout=10m

Inspect the launcher logs. Expect `Workers: 2`, two rank lines, and a calculated pi value. Trainer versions can generate different Job and pod names; selecting the launcher by JobSet labels avoids hard-coding those names.

In [ ]:
!kubectl logs -n {NAMESPACE} -l jobset.sigs.k8s.io/jobset-name=hello-mpi,jobset.sigs.k8s.io/replicatedjob-name=launcher -c node --prefix=true
!kubectl get trainjob hello-mpi -n {NAMESPACE} -o jsonpath='{.status.conditions}'
!kubectl get jobset,pods -n {NAMESPACE}

## Adapt the job to your application

Use your own image for `trainer.image`, make sure it contains the matching OpenMPI implementation and your application dependencies, then update `trainer.command`. Keep an MPI-enabled runtime image for the worker `node` job so it can run SSH and accept connections from the launcher. Request enough CPUs, memory, and accelerators in `resourcesPerNode` for each training node.

For example, a DeepSpeed application can use a command such as:

```yaml
trainer:
  numNodes: 2
  numProcPerNode: 4
  image: registry.example.com/my-deepspeed-image:1.0
  command:
    - mpirun
    - python
    - train.py
```

`numNodes` is the number of MPI training nodes. When `runLauncherAsNode` is true, it includes the launcher. `numProcPerNode` sets the number of slots per node in the generated MPI hostfile. Check the runtime setting before translating old `MPIJob` launcher and worker counts; the conversion is not a field-by-field copy.

## Migrating from MPIJob

In an old `MPIJob`, `mpiReplicaSpecs` describes launcher and worker pods, while `slotsPerWorker` and other MPI settings live on that resource. In Trainer v2, move the reusable pod topology, SSH configuration, and MPI implementation into a `TrainingRuntime` or cluster-wide `ClusterTrainingRuntime`. Put the image, command, node count, and process count for one run in `TrainJob.spec.trainer`.

Trainer v2 does not reconcile the old `kubeflow.org` `MPIJob` resource. Existing `MPIJob` workloads need the MPI Operator that serves that CRD. See the upstream [MPIJob migration guide](https://trainer.kubeflow.org/en/latest/operator-guides/migration.html) for the migration example and launcher-count details.

## Clean up

Delete the sample job when finished. Keep the `TrainingRuntime` if you plan to reuse it for another `TrainJob`; remove it when it is no longer needed.


In [ ]:
!kubectl delete trainjob hello-mpi -n {NAMESPACE}
!kubectl delete trainingruntime openmpi-trainingruntime -n {NAMESPACE}
